# NOTEBOOK: 02_silver_standardize


- Cleans the messy weight_raw column → converts everything to kg as a float
- Cleans the messy unit_price_raw column → converts everything to USD as a float
- Deduplicates vendor names (Sony Corp. + sony corporation → Sony Corporation)

# 0. Imports and Configs

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from delta.tables import DeltaTable
from datetime import datetime
import sys
import os
from pathlib import Path
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

REPO_ROOT = str(Path(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()).parent.parent)
from utils.config import *

In [0]:

PROCESSED_AT = datetime.now().isoformat()

# 1. Clean Product

In [0]:
# Read Bronze tables
df_products = spark.table(BRONZE_PRODUCTS)

## 1.1 Normalize weight to kg

In [0]:
# The weight column has many formats: "12.5 kg", "250 grams", "370gr", "5.8 Kilograms", "32.9 grs", "141 G.", "6.8 g" etc.
# Strategy: extract the numeric value, detect the unit, convert to kg.

def parse_weight_to_kg(raw):
    """
    Parses any weight string and returns a float in kg.
    Handles: kg, g, grams, gr, grs, kilograms, G, Kg, KG etc.
    Returns None if the value cannot be parsed.
    """
    if raw is None or str(raw).strip().lower() in ("none", "nan", ""):
        return None
    
    raw = str(raw).strip().lower()    
    # Remove trailing period — handles "Kg.", "grs.", "g." etc.
    raw = raw.rstrip(".")
    
    # Extract the numeric part (handles: 12.5, 1,8, 12 5 with spaces)
    # Remove currency symbols and text, keep digits, dots, commas
    numeric_str = re.sub(r"[^\d.,]", " ", raw).strip()
    # Handle comma as decimal separator and remove spaces between digits
    numeric_str = numeric_str.replace(",", ".").replace(" ", "")
    
    try:
        value = float(numeric_str)
    except:
        return None
    
    # Detect unit and convert to kg
    if any(unit in raw for unit in ["kilogram", "kilograms"]):
        return round(value, 4)
    elif raw.endswith("kg") or " kg" in raw:
        return round(value, 4)
    elif any(unit in raw for unit in ["gram", "grams", "gr", "grs", "g.", " g"]):
        return round(value / 1000, 4)
    elif raw[-1] == "g" or raw.endswith("kg"):
        # ends with just "g" like "900g" or "370gr"
        if "kg" in raw:
            return round(value, 4)
        else:
            return round(value / 1000, 4)
    else:
        # Default: if unit unclear but value seems reasonable for kg, keep as is
        return round(value, 4)
    
# Register as a Spark UDF so we can apply it to the whole column at once
parse_weight_udf = udf(parse_weight_to_kg, DoubleType())

# Apply the UDF
df_products = df_products.withColumn(
    "weight_kg",
    parse_weight_udf(F.col("weight_raw"))
)

print("✅ Weight normalized")


In [0]:
df_products.display()

## 1.2 Normalize price to USD float

In [0]:
# Normalize price to USD float
# Formats found: "$499.99", "749", "999.00 USD", "59 99", "$4 50.00", "349,00 usd"
# Strategy: strip all non-numeric characters carefully, handle edge cases
def parse_price_to_usd(raw):
    
    if raw is None or str(raw).strip().lower() in ("none", "nan", ""):
        return None
    
    raw = str(raw).strip().lower()
    
    # Remove known text
    raw = raw.replace("$", "").replace("usd", "").replace("dollars", "").strip()
    
    # If both comma AND period exist → comma is thousands separator → remove it
    # Example: "1,099.00" → "1099.00"
    if "," in raw and "." in raw:
        raw = raw.replace(",", "")
    
    # If only comma exists and 2 digits follow → decimal separator
    # Example: "349,00" → "349.00"
    elif "," in raw and "." not in raw:
        parts = raw.split(",")
        if len(parts) == 2 and len(parts[1].strip()) <= 2:
            raw = raw.replace(",", ".")
        else:
            raw = raw.replace(",", "")
    
    # Handle space as decimal separator: "59 99" → "59.99"
    space_decimal = re.match(r"^(\d+)\s(\d{2})$", raw.strip())
    
    if space_decimal:
        raw = f"{space_decimal.group(1)}.{space_decimal.group(2)}"
    else:
        raw = raw.replace(" ", "")
    
    try:
        return round(float(raw), 2)
    except:
        return None

parse_price_udf = udf(parse_price_to_usd, DoubleType())


In [0]:

df_products = df_products.withColumn(
    "unit_price_usd",
    parse_price_udf(F.col("unit_price_raw"))
)

print("✅ Price normalized")
display(df_products.select("product_id", "product_description_raw", "unit_price_raw", "unit_price_usd"))

## 1.3 Normalize Product Description

In [0]:
def normalize_product_description(raw):
    """
    Normalizes product descrition Generic cleaning: 
    """
    #Guard clause: handle None and empty-like values
    if raw is None:
        return None
    
    cleaned = str(raw).strip().lower()
    
    if cleaned == "" or cleaned in ("none", "nan", "null"):
        return None
    
    # ── STEP 1: Generic cleaning ─────────────────────────────────────────
    normalized = " ".join(cleaned.split())   # collapse extra spaces
    normalized = normalized.lower()           # title case

    return normalized

normalize_product_description_udf = udf(normalize_product_description)

df_products = df_products.withColumn(
    "product_description",
    normalize_product_description_udf(F.col("product_description_raw"))
)

print("✅ Vendor names normalized")
display(df_products.select("product_id", "product_description_raw", "product_description"))

## 1.4 Drop duplicates

In [0]:
df_products = df_products.dropDuplicates(["product_id"])

## 1.5 Select and save

In [0]:
# selection of cleaned columns and types adjust

df_products_silver = df_products.select(
    F.col("product_id").cast("integer"),
    F.col("product_description"),
    F.col("weight_kg"),
    F.col("unit_price_usd"),
    F.col("_ingested_at"),
    F.col("_source_system"),
    F.lit(PROCESSED_AT).alias("_processed_at")
)


In [0]:
df_products_silver.display()

In [0]:
# Write Silver tables to Delta
# overwrite + mergeSchema = idempotent and schema-evolution-ready

(df_products_silver.write
    .format("delta")
    .mode("overwrite")
    # .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_PRODUCTS))

In [0]:
# Validate
print("=== VALIDATION ===")
print(f"silver_products row count: {spark.table(SILVER_PRODUCTS).count()}")

# Check for any nulls in critical columns — these would mean our parser failed
null_weights = spark.table(SILVER_PRODUCTS).filter(F.col("weight_kg").isNull()).count()
null_prices  = spark.table(SILVER_PRODUCTS).filter(F.col("unit_price_usd").isNull()).count()

print(f"\nNull weights : {null_weights}")
print(f"Null prices  : {null_prices}")

spark.table(SILVER_PRODUCTS).display()

# 2. Clean Vendors

In [0]:
# Read Bronze tables
df_vendors  = spark.table(BRONZE_VENDORS)

## 2.1 Normalize vendor name

In [0]:
def remove_suffixes(text: str) -> str:
    """
    Removes legal suffixes only when they appear at the END of the string.
    The $ anchor ensures we never remove these patterns from inside words.
    """
    # Order matters — longer suffixes first to avoid partial matches
    suffixes = [
        r"\s+corporation$",   # " corporation" at end
        r"\s+corp\.$",        # " corp." at end
        r"\s+corp$",          # " corp" at end
        r"\s+incorporated$",  # " incorporated" at end
        r"\s+inc\.$",         # " inc." at end
        r"\s+inc$",           # " inc" at end
        r"\s+limited$",       # " limited" at end
        r"\s+ltd\.$",         # " ltd." at end
        r"\s+ltd$",           # " ltd" at end
        r"\s+co\.$",          # " co." at end
        r"\s+co$",            # " co" at end
        r",$",                # trailing comma
    ]

    for suffix in suffixes:
        # re.sub replaces the pattern only if it matches at end of string ($)
        # re.IGNORECASE makes it case-insensitive
        text = re.sub(suffix, "", text, flags=re.IGNORECASE).strip()

    return text

def normalize_vendor(raw):
    if raw is None:
        return None

    cleaned = str(raw).strip()
    if cleaned == "" or cleaned.lower() in ("none", "nan", "null"):
        return None

    # ── STEP 1: Generic cleaning ──────────────────────────────────────────
    normalized = " ".join(cleaned.split()).lower()  # collapse spaces + lowercase

    # Remove legal suffixes safely — only at end of string
    normalized = remove_suffixes(normalized)

    # Remove trailing period and convert to Title Case
    normalized = normalized.rstrip(".").strip()
    normalized = normalized.title()

    # ── STEP 2: Canonical mapping ─────────────────────────────────────────
    canonical_map = {
        "Amazon.Com"             : "Amazon",
        "Asus Tek"               : "Asus",
        "Asus Tek Computer"      : "Asus",
        "Hyperx (Kingston)"      : "HyperX (Kingston)",
        "Hp"                     : "HP",
        "Dji Technology"         : "DJI Technology",
        "Logitech International S.A": "Logitech",
    }

    return canonical_map.get(normalized, normalized)


normalize_vendor_udf = udf(normalize_vendor)

df_vendors = df_vendors.withColumn(
    "vendor_name",
    normalize_vendor_udf(F.col("vendor_name_raw"))
)

print("✅ Vendor names normalized")
display(df_vendors.select("product_id", "vendor_name_raw", "vendor_name"))

In [0]:
display(df_vendors.select("vendor_name").distinct())


## 2.2 Drop Duplicate

In [0]:
df_vendors = df_vendors.dropDuplicates(["product_id","vendor_name"])

## 2.3 Select and Save

In [0]:
# selection of cleaned columns and types adjust
df_vendors_silver = df_vendors.select(
    F.col("product_id").cast("integer"),
    F.col("vendor_name"),
    F.col("vendor_name_raw"),
    F.col("_ingested_at"),
    F.lit(PROCESSED_AT).alias("_processed_at")
)

In [0]:
print("✅ Silver DataFrames ready")
display(df_vendors_silver)

In [0]:

(df_vendors_silver.write
    .format("delta")
    .mode("overwrite")
    # .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_VENDORS))

print(f"✅ Written: {SILVER_VENDORS}")

In [0]:
# Validate
print("=== VALIDATION ===")
print(f"silver_vendors  row count: {spark.table(SILVER_VENDORS).count()}")